# Session 2 — RAG chain with Ollama

**Goal**: connect the retriever (Chroma, session 1) with `llama3.2` and build your first complete RAG chain: **Retrieve → Augment → Generate**.

**Prerequisites**:
- Session 1 ingestion completed (`chroma_db/` folder exists)
- `uv add langchain-ollama`
- Ollama running (app open or `ollama serve`)

## 1. Imports and loading the existing vector store

No re-ingestion needed: we load the Chroma store persisted in session 1.

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

CHROMA_PATH = "chroma_db"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings)

print(f"Vectors in store: {db._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors in store: 5


## 2. The retriever

We turn the vector store into a retriever — the same similarity search as in `verify.py`, but with LangChain's standard interface.

In [15]:
retriever = db.as_retriever(search_kwargs={"k": 1})

# Try it on its own, no LLM yet:
docs = retriever.invoke("What is Chroma used for?")
for d in docs:
    print(d.page_content[:100], "\n---")

Chroma is an open-source vector database that runs locally with no infrastructure required. It store 
---


## 3. The local LLM

`temperature=0` = deterministic answers, ideal for document Q&A.

In [16]:
llm = ChatOllama(model="llama3.2", temperature=0)

# Sanity check: the model responds without any context
response = llm.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


## 4. The prompt with context injection

This is where the RAG magic happens. Note the two placeholders: `{context}` and `{question}`.

In [17]:
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context. If the context doesn't contain the answer, say you don't know.

Context:
{context}

Question: {question}

Answer:""")

## 5. The full chain

Built explicitly so you can see every step: **Retrieve → Augment → Generate**.

In [18]:
def rag_chain(question: str) -> str:
    # 1. Retrieve
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    # 2. Augment
    messages = prompt.invoke({"context": context, "question": question})
    # 3. Generate
    response = llm.invoke(messages)
    return response.content

answer = rag_chain("What is Chroma and why is it popular?")
print(answer)

Chroma is an open-source vector database that runs locally with no infrastructure required, storing embeddings on disk and supporting similarity search using cosine distance. It's a popular choice for prototyping RAG systems because setup takes less than a minute.


## 6. Experiment

The second question is the interesting one: the model should say *\"I don't know\"* instead of making up an answer. That's the prompt doing its job — *grounding*.

In [19]:
print(rag_chain("What are the components of a RAG pipeline?"))

The main components of a RAG pipeline are:

1. Document loader
2. Text splitter
3. Embedding model
4. Vector store
5. Retriever


In [20]:
print(rag_chain("Who won the World Cup in 2010?"))  # <- not in the context

I don't know. The provided context doesn't contain information about sports or specific events like the World Cup.


## Bonus exercise

Change `k=2` to `k=1` and `k=4` in the retriever and re-run the questions. Observe the trade-off:
- too little context → incomplete answers
- too much → noise

Tuning `k` is a real design decision in production systems.

## NOTES
- Chroma is essentially a database whose main query operation is k-nearest neighbors.